In [ ]:
!pip install langchain-core
import os, re
import tiktoken
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

d:\Astro Books\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import re
import os

BOOK_NAMES = {
    "BPHS_English_Book.md":   "Brihat Parashara Hora Shastra (English)",
    "BPHS_Hindi_Book.md":     "Brihat Parashara Hora Shastra (Hindi)",
    "Phaladeepika_part_1.md": "Phaladeepika Part 1",
    "Phaladeepika_part_2.md": "Phaladeepika Part 2",
    "Saravali.md":            "Saravali",
}

def split_by_chapters(text):
    pattern = r'(?=# Chapter \d+)'
    chunks = re.split(pattern, text)
    return [c.strip() for c in chunks if len(c.strip()) > 200]

def remove_toc(text: str) -> str:
    # find the first real "# Chapter" and start from there
    match = re.search(r'\n# Chapter', text)
    if match:
        return text[match.start():]
    return text

def split_documents(docs):
    final_chunks = []

    for doc in docs:
        filename = os.path.basename(doc.metadata.get("source", ""))
        book_name = BOOK_NAMES.get(filename, filename)

        # normalize line endings
        clean_text = doc.page_content.replace('\r\n', '\n').replace('\r', '\n')
        clean_text = remove_toc(clean_text)

        # stage 1: split by chapter
        chapter_chunks = split_by_chapters(clean_text)

        for chapter_text in chapter_chunks:
            # extract chapter title from first line
            first_line = chapter_text.split('\n')[0].strip()
            chapter_title = first_line.replace('#', '').strip()

            from langchain_core.documents import Document
            chunk_doc = Document(
                page_content=chapter_text,
                metadata={
                    "book": book_name,
                    "chapter": chapter_title,
                    "source": doc.metadata.get("source", "")
                }
            )

            # stage 2: re-split if too large
            # in your chapter loop, skip tiny chunks# in your chapter loop, skip tiny chunks
            if len(chapter_text.strip()) < 300:
                continue
            if len(chapter_text) > 700:
                sub_chunks = char_splitter.split_documents([chunk_doc])
                final_chunks.extend(sub_chunks)
            else:
                final_chunks.append(chunk_doc)

    print(f'Total chunks created: {len(final_chunks)}')
    return final_chunks

In [ ]:
# test it quickly
from src.document_loader import load_documents

docs = load_documents()

chunks = split_documents(docs)
print(f"Total: {len(chunks)}")

# check first 20 chunks
for i, c in enumerate(chunks[:20]):
    print(f"\n[{i}] {c.metadata}")
    print(c.page_content[:150])

In [ ]:
for i,c in enumerate(chunks):
    print(
        i,
        len(c.page_content),
        c.metadata
    )